# 实践项目 01：MRI 脑肿瘤图像分割

我们将在 Kaggle Notebook 中读取配对的 MRI 切片与肿瘤掩膜，按患者划分数据，训练轻量 U-Net，并用 Dice 与 IoU 评价预测区域。

## 实践任务
1. 查找 MRI 与 mask 文件并完成一一配对
2. 统计患者数量、阳性 mask 数量与空 mask 比例
3. 按患者划分训练、验证和测试数据
4. 完成同步预处理与数据增强
5. 补全轻量 U-Net 的关键模块
6. 完成训练并保存损失与 Dice 曲线
7. 显示测试图像、真实 mask 和预测 mask

## 需要保存的结果
- `task1_data_check.png`
- `task1_training_curve.png`
- `task1_prediction.png`
- `task1_result.json`


## 需要提交的结果

- `task1_data_check.png`：MRI、mask 与叠加图
- `task1_training_curve.png`：训练与验证损失、Dice
- `task1_prediction.png`：测试样本真实 mask 与预测 mask
- `task1_result.json`：数据规模、参数和最终指标


In [ ]:
from pathlib import Path
import json, random, re
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
print('device:',DEVICE)

## 1. 查找并配对图像与 mask

推荐挂载 `lgg-mri-segmentation` 数据集。图像文件名与 mask 文件名只相差 `_mask`。患者标识从上级文件夹获得。


In [ ]:
ROOT=Path('/kaggle/input')
all_tif=list(ROOT.glob('**/*.tif'))+list(ROOT.glob('**/*.png'))
mask_paths=[p for p in all_tif if '_mask' in p.stem.lower()]
pairs=[]
for m in mask_paths:
    img=Path(str(m).replace('_mask',''))
    if img.exists():
        patient=m.parent.name
        pairs.append((img,m,patient))
print('paired slices:',len(pairs))
assert pairs, '没有找到配对 MRI 与 mask，请查看额外注意事项中的数据集挂载说明。'

## 任务 1：完成数据核对

计算患者数、阳性 mask 数和空 mask 比例。随后显示一张 MRI、mask 和叠加图。


In [ ]:
# TODO 1
patient_count = None
positive_masks = None
empty_ratio = None

sample_img_path, sample_mask_path, _ = pairs[len(pairs)//2]
img=np.array(Image.open(sample_img_path).convert('L'))
mask=np.array(Image.open(sample_mask_path).convert('L'))>0
print(patient_count, positive_masks, empty_ratio, img.shape, mask.shape)

fig,ax=plt.subplots(1,3,figsize=(10,3))
ax[0].imshow(img,cmap='gray'); ax[0].set_title('MRI')
ax[1].imshow(mask,cmap='gray'); ax[1].set_title('mask')
ax[2].imshow(img,cmap='gray'); ax[2].imshow(mask,alpha=.4,cmap='viridis'); ax[2].set_title('overlay')
for a in ax:a.axis('off')
plt.tight_layout(); plt.savefig(OUT/'task1_data_check.png',dpi=160); plt.show()

## 2. 患者级划分

同一患者的切片必须处于同一集合。这里只使用固定数量切片，以便两小时内完成。


In [ ]:
MAX_SLICES=min(1200,len(pairs))
pairs=pairs[:MAX_SLICES]
groups=np.array([x[2] for x in pairs]); idx=np.arange(len(pairs))
gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
train_idx,test_idx=next(gss.split(idx,groups=groups))
train_groups=groups[train_idx]
gss2=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
tr_rel,va_rel=next(gss2.split(train_idx,groups=train_groups))
tr_idx=train_idx[tr_rel]; va_idx=train_idx[va_rel]
assert not (set(groups[tr_idx]) & set(groups[va_idx]) | set(groups[tr_idx]) & set(groups[test_idx]) | set(groups[va_idx]) & set(groups[test_idx]))
print(len(tr_idx),len(va_idx),len(test_idx))

In [ ]:
class MRIDataset(Dataset):
    def __init__(self,pairs,indices,size=128,augment=False):
        self.items=[pairs[i] for i in indices]; self.size=size; self.augment=augment
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        ip,mp,pid=self.items[i]
        image=Image.open(ip).convert('L').resize((self.size,self.size))
        mask=Image.open(mp).convert('L').resize((self.size,self.size),Image.Resampling.NEAREST)
        x=np.asarray(image,dtype=np.float32)/255.0
        y=(np.asarray(mask)>0).astype(np.float32)
        if self.augment and random.random()<.5:
            x=np.fliplr(x).copy(); y=np.fliplr(y).copy()
        return torch.from_numpy(x[None]),torch.from_numpy(y[None]),pid

train_ds=MRIDataset(pairs,tr_idx,augment=True); val_ds=MRIDataset(pairs,va_idx); test_ds=MRIDataset(pairs,test_idx)
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True,num_workers=2)
val_loader=DataLoader(val_ds,batch_size=16,shuffle=False,num_workers=2)
test_loader=DataLoader(test_ds,batch_size=16,shuffle=False,num_workers=2)

## 任务 2：补全 Dice

Dice 比较预测前景与真实前景的重叠。函数输入为概率图和二值标签。


In [ ]:
def dice_score(prob,target,threshold=.5,eps=1e-6):
    # TODO 2：阈值化、计算交集与两侧面积
    return None

## 任务 3：补全 U-Net 卷积块

每个卷积块执行两次 3×3 卷积、归一化与 ReLU。


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,cin,cout):
        super().__init__()
        # TODO 3
        self.block = None
    def forward(self,x): return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1=DoubleConv(1,16); self.p1=nn.MaxPool2d(2)
        self.e2=DoubleConv(16,32); self.p2=nn.MaxPool2d(2)
        self.b=DoubleConv(32,64)
        self.u2=nn.ConvTranspose2d(64,32,2,2); self.d2=DoubleConv(64,32)
        self.u1=nn.ConvTranspose2d(32,16,2,2); self.d1=DoubleConv(32,16)
        self.out=nn.Conv2d(16,1,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p1(e1)); b=self.b(self.p2(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1))
        d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.out(d1)

model=TinyUNet().to(DEVICE)
print('parameters:',sum(p.numel() for p in model.parameters()))

## 任务 4：补全一次训练更新


In [ ]:
bce=nn.BCEWithLogitsLoss()
opt=torch.optim.Adam(model.parameters(),lr=1e-3)

def run_epoch(loader,training):
    model.train(training); losses=[]; dices=[]
    for x,y,_ in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        if training:
            # TODO 4：清梯度
            pass
        logits=model(x); prob=torch.sigmoid(logits)
        loss=bce(logits,y)+(1-dice_score(prob,y))
        if training:
            # TODO 4：反向传播与更新
            pass
        losses.append(float(loss.detach().cpu())); dices.append(float(dice_score(prob,y).detach().cpu()))
    return np.mean(losses),np.mean(dices)

history=[]; best=None; best_d=-1
for epoch in range(3):
    tl,td=run_epoch(train_loader,True); vl,vd=run_epoch(val_loader,False)
    history.append((tl,td,vl,vd)); print(epoch+1,history[-1])
    if vd>best_d: best_d=vd; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best)

In [ ]:
h=np.array(history)
fig,ax=plt.subplots(1,2,figsize=(9,3.5))
ax[0].plot(h[:,0],label='train'); ax[0].plot(h[:,2],label='validation'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(h[:,1],label='train'); ax[1].plot(h[:,3],label='validation'); ax[1].set_title('Dice'); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT/'task1_training_curve.png',dpi=160); plt.show()

## 任务 5：测试评价与阈值比较

分别使用 0.3、0.5 和 0.7 阈值，选择验证 Dice 最高的阈值，再在测试集评价。


In [ ]:
# TODO 5：完成阈值选择
thresholds=[.3,.5,.7]
val_dices={}
# 在此遍历验证集并填写 val_dices
best_threshold=None

def evaluate(loader,threshold):
    model.eval(); ds=[]; ious=[]; examples=[]
    with torch.no_grad():
        for x,y,pid in loader:
            prob=torch.sigmoid(model(x.to(DEVICE))).cpu(); pred=(prob>=threshold).float()
            inter=(pred*y).sum((1,2,3)); union=((pred+y)>0).float().sum((1,2,3))
            d=(2*inter+1e-6)/(pred.sum((1,2,3))+y.sum((1,2,3))+1e-6)
            i=(inter+1e-6)/(union+1e-6)
            ds.extend(d.numpy()); ious.extend(i.numpy())
            if len(examples)<4:
                for k in range(min(4-len(examples),len(x))): examples.append((x[k,0].numpy(),y[k,0].numpy(),prob[k,0].numpy()))
    return float(np.mean(ds)),float(np.mean(ious)),examples

test_dice,test_iou,examples=evaluate(test_loader,best_threshold)
print(best_threshold,test_dice,test_iou)

In [ ]:
fig,ax=plt.subplots(len(examples),3,figsize=(8,2.5*len(examples)))
for r,(x,y,p) in enumerate(examples):
    ax[r,0].imshow(x,cmap='gray'); ax[r,0].set_title('MRI')
    ax[r,1].imshow(y,cmap='gray'); ax[r,1].set_title('true mask')
    ax[r,2].imshow(x,cmap='gray'); ax[r,2].imshow(p>=best_threshold,alpha=.4,cmap='viridis'); ax[r,2].set_title('prediction')
    for c in range(3): ax[r,c].axis('off')
plt.tight_layout(); plt.savefig(OUT/'task1_prediction.png',dpi=160); plt.show()

result={'train_slices':len(train_ds),'validation_slices':len(val_ds),'test_slices':len(test_ds),'best_threshold':best_threshold,'test_dice':test_dice,'test_iou':test_iou,'seed':SEED}
(OUT/'task1_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
result

## 提交说明

用 250–400 字说明数据配对、患者级划分、最佳阈值、测试 Dice/IoU 和一个失败案例。
